# Tier 2 Implementation

In [1]:
import ray
import hashlib
import random
import time
import pprint
from ray.exceptions import RayActorError

if ray.is_initialized():
    ray.shutdown()

try:
    print("Attempting to connect to the Docker Ray cluster...")
    ray.init(address="ray://localhost:10001", ignore_reinit_error=True)
    print("Successfully connected to the external Docker Ray cluster!")
except Exception as e1:
    try:
        ray.init(address="ray://ray-head:10001", ignore_reinit_error=True)
        print("Successfully connected to the internal Docker Ray cluster!")
    except Exception as e2:
        print(f"Could not connect to Docker Ray cluster. Starting standalone local Ray instance...")
        ray.init(ignore_reinit_error=True)
        print("Standalone local Ray instance started successfully!")

print("\n--- RAY CLUSTER RESOURCES ---")
pprint.pprint(ray.cluster_resources())

2026-05-31 20:14:03,989	INFO client_builder.py:241 -- Passing the following kwargs to ray.init() on the server: ignore_reinit_error, log_to_driver


Attempting to connect to the Docker Ray cluster...


/home/ray/anaconda3/lib/python3.10/site-packages/ray/util/client/worker.py:267: UserWarning: Ray Client connection timed out. Ensure that the Ray Client port on the head node is reachable from your local machine. See https://docs.ray.io/en/latest/cluster/ray-client.html#step-2-check-ports for more information.
  warnings.warn(
2026-05-31 20:14:34,013	INFO client_builder.py:241 -- Passing the following kwargs to ray.init() on the server: ignore_reinit_error, log_to_driver
SIGTERM handler is not set because current thread is not the main thread.


Successfully connected to the internal Docker Ray cluster!

--- RAY CLUSTER RESOURCES ---
{'CPU': 20.0,
 'memory': 14427013530.0,
 'node:172.18.0.2': 1.0,
 'node:172.18.0.4': 1.0,
 'node:__internal_head__': 1.0,
 'object_store_memory': 6183005798.0}


## 1. DataNode Actor

In [2]:
@ray.remote
class DataNode:
    def __init__(self, name: str):
        self.name = name
        self.blocks = {}
        self.alive = True

    def ping(self) -> str:
        if not self.alive:
            raise Exception(f"DataNode {self.name} is failed!")
        return "pong"

    def write_block(self, block_id: str, content: str) -> bool:
        if not self.alive:
            raise Exception(f"DataNode {self.name} is failed!")
        self.blocks[block_id] = content
        print(f"[{self.name}] Wrote block '{block_id}' (size: {len(content)})")
        return True

    def read_block(self, block_id: str) -> str:
        if not self.alive:
            raise Exception(f"DataNode {self.name} is failed!")
        if block_id not in self.blocks:
            raise KeyError(f"Block '{block_id}' not found on {self.name}")
        return self.blocks[block_id]

    def delete_block(self, block_id: str) -> bool:
        if not self.alive:
            raise Exception(f"DataNode {self.name} is failed!")
        if block_id in self.blocks:
            del self.blocks[block_id]
            print(f"[{self.name}] Deleted block '{block_id}'")
            return True
        return False

    def list_blocks(self) -> dict:
        if not self.alive:
            raise Exception(f"DataNode {self.name} is failed!")
        return {bid: len(content) for bid, content in self.blocks.items()}

    def get_name(self) -> str:
        return self.name

    def replicate_block_to(self, block_id: str, target_dn_handle) -> bool:
        if not self.alive:
            raise Exception(f"DataNode {self.name} is failed!")
        if block_id not in self.blocks:
            raise KeyError(f"Block '{block_id}' not found on {self.name} for replication")
        
        content = self.blocks[block_id]
        print(f"[{self.name}] Replicating block '{block_id}' directly to target node...")
        return ray.get(target_dn_handle.write_block.remote(block_id, content))

## 2. NameNode Actor

In [3]:
@ray.remote
class NameNode:
    def __init__(self, replication_factor: int = 3, block_size: int = 16):
        self.replication_factor = replication_factor
        self.block_size = block_size
        self.datanodes = {}
        self.datanodes_status = {}
        self.artifacts = {}
        
    def register_datanode(self, name: str, handle) -> str:
        self.datanodes[name] = handle
        self.datanodes_status[name] = "ALIVE"
        print(f"[NameNode] Registered DataNode '{name}'")
        return f"DataNode {name} registered."
        
    def get_active_datanodes(self) -> list:
        active = []
        for name, handle in self.datanodes.items():
            if self.datanodes_status[name] == "ALIVE":
                active.append(name)
        return active

    def get_block_size(self) -> int:
        return self.block_size

    def prepare_upload(self, artifact_name: str, num_blocks: int) -> list:
        active_nodes = self.get_active_datanodes()
        if not active_nodes:
            raise Exception("No active DataNodes available!")
            
        block_allocations = []
        self.artifacts[artifact_name] = []
        
        for i in range(num_blocks):
            block_id = f"{artifact_name}_block_{i}"
            k = min(self.replication_factor, len(active_nodes))
            chosen = random.sample(active_nodes, k)
            
            block_meta = {
                "block_id": block_id,
                "hash": "",
                "locations": chosen
            }
            self.artifacts[artifact_name].append(block_meta)
            
            block_allocations.append({
                "block_id": block_id,
                "block_index": i,
                "targets": {node_name: self.datanodes[node_name] for node_name in chosen}
            })
            
        print(f"[NameNode] Prepared upload for '{artifact_name}' with {num_blocks} blocks.")
        return block_allocations

    def get_artifact_metadata(self, artifact_name: str) -> dict:
        if artifact_name not in self.artifacts:
            return None
            
        meta_list = []
        for block in self.artifacts[artifact_name]:
            meta_list.append({
                "block_id": block["block_id"],
                "hash": block["hash"],
                "locations": block["locations"],
                "handles": {node_name: self.datanodes[node_name] for node_name in block["locations"] if self.datanodes_status[node_name] == "ALIVE"}
            })
        return {
            "artifact_name": artifact_name,
            "block_size": self.block_size,
            "blocks": meta_list
        }

    def update_block_hash(self, artifact_name: str, block_index: int, block_hash: str):
        if artifact_name in self.artifacts and block_index < len(self.artifacts[artifact_name]):
            self.artifacts[artifact_name][block_index]["hash"] = block_hash

    def prepare_block_update(self, artifact_name: str, block_index: int) -> dict:
        if artifact_name not in self.artifacts:
            raise KeyError(f"Artifact '{artifact_name}' not initialized.")
            
        while block_index >= len(self.artifacts[artifact_name]):
            block_id = f"{artifact_name}_block_{len(self.artifacts[artifact_name])}"
            self.artifacts[artifact_name].append({
                "block_id": block_id,
                "hash": "",
                "locations": []
            })
            
        block = self.artifacts[artifact_name][block_index]
        block_id = block["block_id"]
        
        healthy_locations = [loc for loc in block["locations"] if self.datanodes_status[loc] == "ALIVE"]
        
        needed = self.replication_factor - len(healthy_locations)
        if needed > 0:
            active_nodes = self.get_active_datanodes()
            candidates = [node for node in active_nodes if node not in healthy_locations]
            added = random.sample(candidates, min(needed, len(candidates)))
            healthy_locations.extend(added)
            
        block["locations"] = healthy_locations
        
        print(f"[NameNode] Prepared update for block {block_id} on {healthy_locations}")
        return {
            "block_id": block_id,
            "block_index": block_index,
            "targets": {node_name: self.datanodes[node_name] for node_name in healthy_locations}
        }

    def truncate_artifact_blocks(self, artifact_name: str, new_num_blocks: int) -> bool:
        if artifact_name not in self.artifacts:
            return False
            
        old_blocks = self.artifacts[artifact_name]
        keep_blocks = old_blocks[:new_num_blocks]
        delete_blocks = old_blocks[new_num_blocks:]
        
        self.artifacts[artifact_name] = keep_blocks
        
        for block in delete_blocks:
            block_id = block["block_id"]
            for node_name in block["locations"]:
                if self.datanodes_status[node_name] == "ALIVE":
                    self.datanodes[node_name].delete_block.remote(block_id)
                    
        print(f"[NameNode] Truncated '{artifact_name}' to {new_num_blocks} blocks.")
        return True

    def delete_artifact(self, artifact_name: str) -> bool:
        if artifact_name not in self.artifacts:
            return False
        blocks_to_delete = self.artifacts[artifact_name]
        del self.artifacts[artifact_name]
        
        for block in blocks_to_delete:
            block_id = block["block_id"]
            for node_name in block["locations"]:
                if self.datanodes_status[node_name] == "ALIVE":
                    self.datanodes[node_name].delete_block.remote(block_id)
                    
        print(f"[NameNode] Deleted all physical blocks and metadata for '{artifact_name}'.")
        return True

    def check_health_and_replicate(self) -> dict:
        status_updates = {}
        for name, handle in self.datanodes.items():
            if self.datanodes_status[name] == "DEAD":
                continue
                
            try:
                ray.get(handle.ping.remote(), timeout=2.0)
            except Exception as e:
                print(f"[NameNode] Detected DataNode '{name}' FAILURE!")
                self.datanodes_status[name] = "DEAD"
                status_updates[name] = "DEAD"
                
        replications_triggered = []
        active_nodes = self.get_active_datanodes()
        
        for art_name, blocks in self.artifacts.items():
            for block in blocks:
                block_id = block["block_id"]
                original_locations = block["locations"]
                healthy_locations = [loc for loc in original_locations if self.datanodes_status[loc] == "ALIVE"]
                block["locations"] = healthy_locations
                
                current_repl = len(healthy_locations)
                if current_repl < self.replication_factor and current_repl > 0:
                    needed = self.replication_factor - current_repl
                    candidates = [node for node in active_nodes if node not in healthy_locations]
                    
                    if candidates:
                        targets = random.sample(candidates, min(needed, len(candidates)))
                        source_node = healthy_locations[0]
                        source_handle = self.datanodes[source_node]
                        
                        for target in targets:
                            target_handle = self.datanodes[target]
                            print(f"[NameNode] Triggering replication: copy '{block_id}' from {source_node} to {target}")
                            try:
                                success = ray.get(source_handle.replicate_block_to.remote(block_id, target_handle))
                                if success:
                                    block["locations"].append(target)
                                    replications_triggered.append({
                                        "block_id": block_id,
                                        "from": source_node,
                                        "to": target,
                                        "status": "SUCCESS"
                                    })
                            except Exception as rep_err:
                                replications_triggered.append({
                                    "block_id": block_id,
                                    "from": source_node,
                                    "to": target,
                                    "status": f"FAILED: {str(rep_err)}"
                                })
                                
        return {
            "status_updates": status_updates,
            "replications": replications_triggered
        }

    def get_system_status(self) -> dict:
        nodes_info = {}
        for name in self.datanodes:
            status = self.datanodes_status[name]
            blocks_stored = []
            if status == "ALIVE":
                try:
                    blocks_stored = list(ray.get(self.datanodes[name].list_blocks.remote()).keys())
                except Exception:
                    blocks_stored = ["Error fetching blocks"]
            nodes_info[name] = {
                "status": status,
                "blocks_stored": blocks_stored
            }
            
        artifacts_info = {}
        for art_name, blocks in self.artifacts.items():
            blocks_info = []
            for block in blocks:
                blocks_info.append({
                    "block_id": block["block_id"],
                    "hash": block["hash"][:8] if block["hash"] else "None",
                    "locations": block["locations"]
                })
            artifacts_info[art_name] = blocks_info
            
        return {
            "datanodes": nodes_info,
            "artifacts": artifacts_info
        }

## 3. HDFSClient Class

In [ ]:
class HDFSClient:
    def __init__(self, namenode_handle):
        self.namenode = namenode_handle
        self.block_size = ray.get(self.namenode.get_block_size.remote())
        
    def _compute_hash(self, content: str) -> str:
        return hashlib.sha256(content.encode("utf-8")).hexdigest()
        
    def upload(self, artifact_name: str, content: str):
        chunks = [content[i:i+self.block_size] for i in range(0, len(content), self.block_size)]
        num_blocks = len(chunks)
        
        allocations = ray.get(self.namenode.prepare_upload.remote(artifact_name, num_blocks))
        
        for alloc in allocations:
            block_id = alloc["block_id"]
            idx = alloc["block_index"]
            chunk_content = chunks[idx]
            targets = alloc["targets"]
            
            for node_name, node_handle in targets.items():
                try:
                    ray.get(node_handle.write_block.remote(block_id, chunk_content))
                except Exception as e:
                    print(f"[Client] Warning: Failed to write {block_id} to {node_name}: {e}")
            
            block_hash = self._compute_hash(chunk_content)
            self.namenode.update_block_hash.remote(artifact_name, idx, block_hash)
            
        print(f"[Client] Uploaded '{artifact_name}' successfully ({num_blocks} blocks).")
        
    def download(self, artifact_name: str) -> str:
        meta = ray.get(self.namenode.get_artifact_metadata.remote(artifact_name))
        if not meta:
            raise KeyError(f"Artifact '{artifact_name}' not found!")
            
        blocks_data = []
        for block in meta["blocks"]:
            block_id = block["block_id"]
            handles = block["handles"]
            
            success = False
            for node_name, node_handle in handles.items():
                try:
                    content = ray.get(node_handle.read_block.remote(block_id), timeout=2.0)
                    blocks_data.append(content)
                    success = True
                    break
                except Exception as e:
                    print(f"[Client] Failover: Failed to read {block_id} from {node_name}. Trying next replica...")
            
            if not success:
                raise RuntimeError(f"Failed to read block {block_id} from any active DataNodes!")
                
        return "".join(blocks_data)
        
    def update(self, artifact_name: str, content: str):
        meta = ray.get(self.namenode.get_artifact_metadata.remote(artifact_name))
        if not meta:
            print(f"[Client] Artifact '{artifact_name}' does not exist. Uploading as fresh.")
            self.upload(artifact_name, content)
            return
            
        chunks = [content[i:i+self.block_size] for i in range(0, len(content), self.block_size)]
        new_num_blocks = len(chunks)
        old_num_blocks = len(meta["blocks"])
        
        for i in range(max(new_num_blocks, old_num_blocks)):
            if i < new_num_blocks:
                new_chunk = chunks[i]
                new_hash = self._compute_hash(new_chunk)
                
                if i < old_num_blocks and meta["blocks"][i]["hash"] == new_hash:
                    print(f"[Client] Block '{artifact_name}_block_{i}' UNCHANGED (differential update skip)")
                    continue
                
                action = "Updating" if i < old_num_blocks else "Appending"
                print(f"[Client] {action} block '{artifact_name}_block_{i}' (differential update write)...")
                
                alloc = ray.get(self.namenode.prepare_block_update.remote(artifact_name, i))
                targets = alloc["targets"]
                block_id = alloc["block_id"]
                
                for node_name, node_handle in targets.items():
                    try:
                        ray.get(node_handle.write_block.remote(block_id, new_chunk))
                    except Exception as e:
                        print(f"[Client] Warning: Failed to write {block_id} to {node_name}: {e}")
                        
                self.namenode.update_block_hash.remote(artifact_name, i, new_hash)
                
        if new_num_blocks < old_num_blocks:
            ray.get(self.namenode.truncate_artifact_blocks.remote(artifact_name, new_num_blocks))
                    
        print(f"[Client] Updated '{artifact_name}' successfully using differential updates.")
        
    def delete(self, artifact_name: str):
        success = ray.get(self.namenode.delete_artifact.remote(artifact_name))
        if success:
            print(f"[Client] Deleted artifact '{artifact_name}' from the system.")
        else:
            print(f"[Client] Artifact '{artifact_name}' not found.")

## 4. System Validation and Test Scenarios

### Test 1: System Initialization

In [5]:
print("Initializing distributed cluster nodes...")
namenode = NameNode.remote(replication_factor=3, block_size=16)

datanodes = {}
for i in range(1, 5):
    name = f"DataNode-{i}"
    handle = DataNode.remote(name)
    datanodes[name] = handle
    ray.get(namenode.register_datanode.remote(name, handle))

client = HDFSClient(namenode)
print("Cluster ready!")

Initializing distributed cluster nodes...
Cluster ready!


### Test 2: Uploading a Large Artifact
We split the string into 16-character blocks and distribute them with a replication factor of 3. We print the system status after upload.

In [6]:
large_string = "abcdefghijklmnopqrstuvwxyz1234567890ABCDEFGHIJKLMNOPQRSTUVWXYZ"

print(f"Uploading artifact 'art-1' (size: {len(large_string)} chars)...")
client.upload("art-1", large_string)

print("\n--- CURRENT CLUSTER STATE ---")
pprint.pprint(ray.get(namenode.get_system_status.remote()))

Uploading artifact 'art-1' (size: 62 chars)...
(NameNode pid=193, ip=172.18.0.4) [NameNode] Registered DataNode 'DataNode-1'
(NameNode pid=193, ip=172.18.0.4) [NameNode] Registered DataNode 'DataNode-2'
(NameNode pid=193, ip=172.18.0.4) [NameNode] Registered DataNode 'DataNode-3'
(NameNode pid=193, ip=172.18.0.4) [NameNode] Registered DataNode 'DataNode-4'
(NameNode pid=193, ip=172.18.0.4) [NameNode] Prepared upload for 'art-1' with 4 blocks.
(DataNode pid=822) [DataNode-4] Wrote block 'art-1_block_0' (size: 16)
(DataNode pid=822) [DataNode-4] Wrote block 'art-1_block_1' (size: 16)
(DataNode pid=822) [DataNode-4] Wrote block 'art-1_block_2' (size: 16)
(DataNode pid=823) [DataNode-2] Wrote block 'art-1_block_0' (size: 16)
(DataNode pid=823) [DataNode-2] Wrote block 'art-1_block_1' (size: 16)
(DataNode pid=823) [DataNode-2] Wrote block 'art-1_block_2' (size: 16)
(DataNode pid=194, ip=172.18.0.4) [DataNode-1] Wrote block 'art-1_block_0' (size: 16)
(DataNode pid=194, ip=172.18.0.4) [DataNo

### Test 3: Downloading the Artifact

In [7]:
print("Downloading artifact 'art-1'...")
downloaded = client.download("art-1")
print(f"Downloaded content: {downloaded}")

assert downloaded == large_string, "Verification FAILED! Content mismatch."
print("Verification SUCCESSFUL: Downloaded content matches original!")

(DataNode pid=194, ip=172.18.0.4) [DataNode-1] Wrote block 'art-1_block_3' (size: 14)
(DataNode pid=275, ip=172.18.0.4) [DataNode-3] Wrote block 'art-1_block_2' (size: 16)
(DataNode pid=275, ip=172.18.0.4) [DataNode-3] Wrote block 'art-1_block_3' (size: 14)
Downloaded content: abcdefghijklmnopqrstuvwxyz1234567890ABCDEFGHIJKLMNOPQRSTUVWXYZ
Verification SUCCESSFUL: Downloaded content matches original!


### Test 4: Updates

In [8]:
modified_string = "abcdefghijklmnopqrstuvwxyz123X567890ABCDEFGHIJKLMNOPQRSTUVWXYZ"

print(f"Modifying string (changing 1 char at index 29)...")
client.update("art-1", modified_string)

print("\nDownloading updated artifact to verify...")
downloaded_updated = client.download("art-1")
print(f"Reconstructed: {downloaded_updated}")

assert downloaded_updated == modified_string, "Update Verification FAILED!"
print("Verification SUCCESSFUL: Incremental update works flawlessly!")

Modifying string (changing 1 char at index 29)...
[Client] Block 'art-1_block_0' UNCHANGED (differential update skip)
[Client] Updating block 'art-1_block_1' (differential update write)...
[Client] Block 'art-1_block_2' UNCHANGED (differential update skip)
[Client] Block 'art-1_block_3' UNCHANGED (differential update skip)
[Client] Updated 'art-1' successfully using differential updates.

Reconstructed: abcdefghijklmnopqrstuvwxyz123X567890ABCDEFGHIJKLMNOPQRSTUVWXYZ
Verification SUCCESSFUL: Incremental update works flawlessly!


### Test 5: High Availability & Failover

In [9]:
meta = ray.get(namenode.get_artifact_metadata.remote("art-1"))
block_0_nodes = meta["blocks"][0]["locations"]
print(f"Block 0 is replica-stored on: {block_0_nodes}")

crashed_node = block_0_nodes[0]
crashed_handle = datanodes[crashed_node]

print(f"Simulating hardware failure (ray.kill) on node: {crashed_node}...")
ray.kill(crashed_handle)

print("\nDownloading immediately after failure (verifying client failover)...")
downloaded_failover = client.download("art-1")
print(f"Downloaded: {downloaded_failover}")

assert downloaded_failover == modified_string, "Failover download failed!"
print("Verification SUCCESSFUL: Client safely bypassed dead node using active replicas!")

Block 0 is replica-stored on: ['DataNode-2', 'DataNode-1', 'DataNode-4']
Simulating hardware failure (ray.kill) on node: DataNode-2...
(DataNode pid=822) [DataNode-4] Wrote block 'art-1_block_1' (size: 16)
(DataNode pid=823) [DataNode-2] Wrote block 'art-1_block_1' (size: 16)

[Client] Failover: Failed to read art-1_block_0 from DataNode-2. Trying next replica...
(NameNode pid=193, ip=172.18.0.4) [NameNode] Prepared update for block art-1_block_1 on ['DataNode-2', 'DataNode-1', 'DataNode-4']
(DataNode pid=194, ip=172.18.0.4) [DataNode-1] Wrote block 'art-1_block_1' (size: 16)
[Client] Failover: Failed to read art-1_block_1 from DataNode-2. Trying next replica...
Downloaded: abcdefghijklmnopqrstuvwxyz123X567890ABCDEFGHIJKLMNOPQRSTUVWXYZ
Verification SUCCESSFUL: Client safely bypassed dead node using active replicas!


### Test 6: Self-Healing & Automatic Replication

In [10]:
print("Triggering periodic healthcheck & self-healing re-replication...")
healing_report = ray.get(namenode.check_health_and_replicate.remote())

print("\n--- self-healing report ---")
pprint.pprint(healing_report)

print("\n--- POST-HEALING CLUSTER STATE ---")
pprint.pprint(ray.get(namenode.get_system_status.remote()))

final_download = client.download("art-1")
assert final_download == modified_string, "Data corruption post replication!"
print("\nVerification SUCCESSFUL: Cluster healed itself and replication is restored!")

Triggering periodic healthcheck & self-healing re-replication...

--- self-healing report ---
{'replications': [{'block_id': 'art-1_block_0',
                   'from': 'DataNode-1',
                   'status': 'SUCCESS',
                   'to': 'DataNode-3'},
                  {'block_id': 'art-1_block_1',
                   'from': 'DataNode-1',
                   'status': 'SUCCESS',
                   'to': 'DataNode-3'},
                  {'block_id': 'art-1_block_2',
                   'from': 'DataNode-4',
                   'status': 'SUCCESS',
                   'to': 'DataNode-1'}],
 'status_updates': {'DataNode-2': 'DEAD'}}

--- POST-HEALING CLUSTER STATE ---
{'artifacts': {'art-1': [{'block_id': 'art-1_block_0',
                          'hash': 'f39dac6c',
                          'locations': ['DataNode-1',
                                        'DataNode-4',
                                        'DataNode-3']},
                         {'block_id': 'art-1_block_1',

### Test 7: Artifact Deletion
We delete the artifact. All physical blocks must be deleted from all DataNodes and the metadata must be wiped.

In [11]:
print("Deleting artifact 'art-1'...")
client.delete("art-1")

print("\n--- CLUSTER STATE POST-DELETION ---")
pprint.pprint(ray.get(namenode.get_system_status.remote()))

ray.shutdown()
print("\nRay shutdown. System testing complete!")

Deleting artifact 'art-1'...
(DataNode pid=822) [DataNode-4] Replicating block 'art-1_block_2' directly to target node...
[Client] Deleted artifact 'art-1' from the system.

--- CLUSTER STATE POST-DELETION ---
{'artifacts': {},
 'datanodes': {'DataNode-1': {'blocks_stored': [], 'status': 'ALIVE'},
               'DataNode-2': {'blocks_stored': [], 'status': 'DEAD'},
               'DataNode-3': {'blocks_stored': [], 'status': 'ALIVE'},
               'DataNode-4': {'blocks_stored': [], 'status': 'ALIVE'}}}
(NameNode pid=193, ip=172.18.0.4) [NameNode] Detected DataNode 'DataNode-2' FAILURE!
(NameNode pid=193, ip=172.18.0.4) [NameNode] Triggering replication: copy 'art-1_block_0' from DataNode-1 to DataNode-3
(NameNode pid=193, ip=172.18.0.4) [NameNode] Triggering replication: copy 'art-1_block_1' from DataNode-1 to DataNode-3
(NameNode pid=193, ip=172.18.0.4) [NameNode] Triggering replication: copy 'art-1_block_2' from DataNode-4 to DataNode-1
(NameNode pid=193, ip=172.18.0.4) [NameNode